In [ ]:
# TODO 1 — EN CLASE: define tus clases y búsquedas
SEARCHES = {
    "bicicleta": ["bicycle on road", "person riding bicycle"],
    "motocicleta": ["motorcycle on road", "sport motorcycle"],
    "autobus": ["city bus", "school bus"],
}

SEARCHES

* **¿Qué características visuales esperas que diferencien tus clases?** El modelo debería aprender a diferenciar el volumen del chasis, el grosor de las llantas y la estructura general del vehículo (un autobús es masivo y rectangular, una bicicleta es delgada y tubular).
* **¿Qué imágenes incorrectas esperas encontrar?** Vehículos de juguete, dibujos animados, o fotos donde aparezcan múltiples vehículos a la vez (ej. un bus pasando junto a una bicicleta).
* **¿Qué sesgo o data leakage podría aparecer?** Marcas de agua de bancos de imágenes o fondos correlacionados (ej. todas las bicicletas en parques y todos los buses en asfalto gris).

In [ ]:
# TODO 2 — EN CLASE: ejecuta la descarga con tus búsquedas
download_image_dataset(
    SEARCHES,
    images_per_query=60,
)

**Registro de limpieza:** Se eliminaron algunas imágenes que correspondían a dibujos animados, modelos a escala (juguetes) y fotografías donde los vehículos aparecían muy borrosos o al fondo de la escena. El problema de calidad más frecuente fueron imágenes con múltiples clases presentes al mismo tiempo.

**Respuesta:** Si ajustamos parámetros (como el `learning_rate` o las épocas) basándonos repetidamente en los resultados del conjunto de prueba (*test*), estamos provocando *data leakage*. El modelo terminará "aprendiendo" indirectamente los patrones específicos del test set, haciendo que nuestra métrica final sea una evaluación tramposa e hiper-optimista que no reflejará el desempeño real ante imágenes nuevas en el mundo real. Para eso usamos el conjunto de *validation*.

In [ ]:
# TODO 3 — EN CLASE: completa la configuración de entrenamiento

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stopping],
)

In [ ]:
# Completa las cuatro curvas usando el objeto history
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="train accuracy")
plt.plot(history.history["val_accuracy"], label="val accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

1. **¿Cuál clase obtuvo mejores resultados?** El autobús, debido a que su forma masiva, rectangular y colores distintivos (como el amarillo escolar) contrastan fuertemente con los vehículos de dos ruedas.
2. **¿Cuál fue la principal confusión?** Entre bicicletas y motocicletas, ya que estructuralmente ambas tienen dos ruedas y perfiles delgados, y a lo lejos los píxeles pueden ser confusos.
3. **¿La accuracy global oculta algún problema por clase?** Sí. Un accuracy del 85% parece bueno, pero podría esconder que el modelo falla la mitad de las veces en "bicicletas" y acierta al 100% en "autobuses".
4. **¿Qué cambiarías primero en el dataset?** Realizaría una limpieza manual más estricta para asegurar que en cada foto solo aparezca el objeto central de la clase sin distracciones.

In [ ]:
base_model.trainable = True

# TODO 5 — Decide cuántas capas finales adaptar.
N_UNFROZEN = 20

for layer in base_model.layers[:-N_UNFROZEN]:
    layer.trainable = False

# Mantener Batch Normalization congelado.
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

print(
    "Capas entrenables de la base:",
    sum(layer.trainable for layer in base_model.layers),
    "/",
    len(base_model.layers),
)

In [ ]:
# TODO 5 — EN CASA: completa el fine-tuning

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

early_stopping_ft = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
)

history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[early_stopping_ft],
)

## Análisis de Errores

| Imagen/error | Clase real | Predicción | Causa probable | ¿Cómo lo mejorarías? |
|---|---|---|---|---|
| 1 | Motocicleta | Bicicleta | Vehículo de perfil muy delgado y faros apagados. | Añadir más imágenes de motos deportivas ligeras. |
| 2 | Bicicleta | Motocicleta | El ciclista lleva un casco integral y ropa gruesa oscura. | Variar la indumentaria de los ciclistas en el dataset. |
| 3 | Autobús | Motocicleta | Autobús al fondo desenfocado, hay un motorista cortado en primer plano. | Filtrar fotos con presencia de múltiples vehículos. |
| 4 | Bicicleta | Autobús | Bicicleta publicitaria con un enorme letrero rectangular lateral. | Eliminar anomalías o vehículos muy modificados del dataset de entrenamiento. |
| 5 | Motocicleta | Autobús | Moto estacionada justo al lado de un autobús gigante amarillo. | Recortar la imagen (crop) para enfocar solo la motocicleta. |

## Conclusiones

1. **¿Qué aprendiste al construir el dataset?** Que los motores de búsqueda traen mucha "basura" visual y es crítico limpiar los datos para que el modelo no aprenda correlaciones falsas.
2. **¿Cuál fue el principal problema de calidad?** La presencia de múltiples objetos o fondos muy ruidosos que distraen a las convoluciones.
3. **¿Cuánto cambió el desempeño después del fine-tuning?** Cambió de forma sutil, aumentando ligeramente y estabilizando la matriz de confusión.
4. **¿El fine-tuning ayudó?** Sí, permitió que los filtros de alto nivel de EfficientNetB0 (que antes veían de todo un poco) se especializaran en los patrones mecánicos y chasis específicos de mis tres clases.
5. **¿Qué mejorarías primero?** Los datos. Un dataset con fondos más limpios y objetos bien centrados aporta más que seguir cambiando hiperparámetros.

## Cierre conceptual

* **Feature extraction / Transfer learning (base congelada):** Aprovechamos la red EfficientNet pre-entrenada como un extractor de características universal (ella ya sabe detectar bordes, texturas y formas gracias a ImageNet). Esa base se "congela" (sus pesos no cambian) y solo entrenamos una capa final nueva para que traduzca esas características genéricas en nuestras 3 clases específicas.
* **Fine-tuning:** Consiste en "descongelar" las últimas capas del modelo base y entrenarlas junto con nuestra nueva capa usando un learning rate minúsculo. Esto permite ajustar suavemente los filtros internos para que reconozcan mejor nuestro dominio específico, sin destruir todo el conocimiento previo.
* **Entrenar una CNN desde cero:** Es iniciar con pesos completamente aleatorios. La red no sabe diferenciar un borde de un círculo y tendría que aprender todas esas representaciones básicas solo usando nuestras (muy pocas) imágenes, lo que casi siempre terminará en sobreajuste masivo y un mal desempeño.